# 2. Amputee Reality: Full-Window Frequency-Domain Action Recognition

## Research Goal for Prosthetic Design

This notebook measures how well residual-limb (amputee) EMG patterns match the ability-bodied baseline from **Notebook 1**. We ask: **Can we recognize hand actions from amputee forearm EMG using the same frequency-domain pipeline?**

**Why this matters:** In prosthetic control, we cannot observe intended actions directly. We infer them from residual-limb muscle activity. The gap between:

- **Notebook 1 baseline:** Action recognition from intact glove kinematics (what actions *look* like).
- **This notebook:** Action recognition from amputee EMG (what actions *feel* like in the residual limb).

...quantifies the **translation challenge** for device design.

**Non-clinical scope:** Offline analysis, not a validated device. Results guide design constraints for prosthetic control research.

## 2. Dataset Characterization: Amputee Cohort (Ninapro DB3)

**Source:** Ninapro DB3 Exercises B & C from transradial amputee participants.

**Measurement:**
- `restimulus`: labeled intended action (0=Rest, 1-17=Exercise B, 18-40=Exercise C)—the same label set as Notebook 1.
- `glove`: 22-channel CyberGlove (movements of prosthetic hand, worn during experiment to guide motor intent).
- **Note:** For this notebook, we use **EMG channels only**, not the glove.
- `rerepetition`: repetition index (1-6), used to prevent train/test leakage.

**Window parameters:** Identical to Notebook 1 for direct comparison.
- 400-sample EMG windows with 100-sample stride.
- Windows with >80% same-action label (purity) are retained.
- Windows crossing a repetition boundary are discarded.
- Frequency representation: log-magnitude FFT, 32 bins per channel = ~384 features (12 EMG channels).

**Why EMG in amputees:** EMG captures residual-limb muscle activity. A prosthetic controller needs to decode intended actions from EMG alone, since the hand is no longer present. This notebook simulates that constraint.

## 3. Setup

- Shared utilities imported from `utils.py`.
- Reproducibility: fixed random seed, group-aware train/test split.
- Same hierarchical model as Notebook 1 for fair comparison.
- PyTorch for consistent training pipeline.

In [ ]:
# Reserve names for later
train_df_amputee = None
test_df_amputee = None

# Import core analysis libraries
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# Import from shared utilities
import sys
sys.path.insert(0, str(Path.cwd().parent))
from utils import (
    ACTIONS, EXERCISE_NAMES,
    set_seed, find_repo_root, get_data_root,
    load_mat_recording,
    make_frequency_windows,
    subject_grouped_holdout, cap_and_shuffle_classes, exercise_labels_from_classes,
    FrequencyMLP, fit_frequency_model, predict_hierarchical,
    classification_summary, save_experiment_json,
    plot_bar, plot_confusion_heatmap,
)

# Import PyTorch
import torch
from sklearn.metrics import classification_report, confusion_matrix

# Matplotlib style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ Core libraries imported")
print(f"✓ PyTorch device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## 4. Configuration and Seeds

In [ ]:
SEED = 42
EXPERIMENT_ID = "EXP-009-amputee-frequency-reality"
COHORT = "Amputee"

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
repo_root = find_repo_root()
data_root = get_data_root(repo_root)
results_dir = repo_root / 'results' / 'metrics'
results_dir.mkdir(parents=True, exist_ok=True)

print(f"SEED: {SEED}")
print(f"Device: {device}")
print(f"Data root: {data_root}")
print(f"Results dir: {results_dir}")

## 5. Data Loading and Windowing

Load all Amputee recordings, create frequency windows from EMG data.

In [ ]:
# Locate all Amputee recordings
amputee_dir = data_root / 'Amputee'
amputee_paths = sorted(amputee_dir.glob('S*_E*.mat'))
print(f"Found {len(amputee_paths)} Amputee recordings")

# Load and window all recordings
records = []
for path in amputee_paths:
    recording = load_mat_recording(path)
    windowed = make_frequency_windows(
        recording, path, cohort='Amputee',
        window_samples=400, stride_samples=100, fft_bins=32, min_purity=0.80
    )
    records.append(windowed)
    print(f"  {path.name}: {windowed[0].shape[0]} windows")

# Concatenate all windows and labels
X = np.concatenate([r[0] for r in records], axis=0)
target_names = np.concatenate([r[1] for r in records])
groups = np.concatenate([r[2] for r in records])

print(f"\nTotal windows: {X.shape[0]}")
print(f"Feature dimension: {X.shape[1]}")
print(f"Unique actions: {len(np.unique(target_names))}")

## 6. Train/Test Split (Group-Aware)

Hold out subjects and their repetitions, matching Notebook 1's strategy.

In [ ]:
# Extract subject IDs from group strings
subjects = np.asarray([g.split(':', 1)[0] for g in groups])
unique_subjects = np.unique(subjects)
print(f"Unique subjects: {len(unique_subjects)}")

# Group-aware holdout split
train_mask, test_mask, test_subjects, val_subjects = subject_grouped_holdout(
    groups, subjects, SEED, test_fraction=0.25
)

print(f"Train windows: {train_mask.sum()}")
print(f"Test windows: {test_mask.sum()}")
print(f"Test subjects: {sorted(test_subjects)}")
print(f"Val subjects: {sorted(val_subjects)}")

# Create class indices
classes = sorted(np.unique(target_names))
y = np.asarray([classes.index(t) for t in target_names])
print(f"\nClass mapping: {len(classes)} classes")

## 7. Training Data Preparation

In [ ]:
# Balance training classes
selected_indices = cap_and_shuffle_classes(y, train_mask, SEED, max_per_class=1000)

X_train = X[selected_indices]
y_train = y[selected_indices]
target_names_train = target_names[selected_indices]

X_test = X[test_mask]
y_test = y[test_mask]
target_names_test = target_names[test_mask]

print(f"Training windows (balanced): {X_train.shape[0]}")
print(f"Test windows: {X_test.shape[0]}")

# Exercise-level labels for hierarchical model
exercise_ids_train, _, y_exercise_train = exercise_labels_from_classes(classes, y_train)
exercise_ids_test, _, y_exercise_test = exercise_labels_from_classes(classes, y_test)

print(f"\nExercises in training: {len(exercise_ids_train)}")
print(f"Exercises in test: {len(exercise_ids_test)}")

## 8. Model Training: Hierarchical Exercise Router + Action Classifiers

In [ ]:
# Train exercise-level router
print("Training exercise router...")
exercise_model = fit_frequency_model(
    X_train, y_exercise_train, len(exercise_ids_train), device,
    epochs=200, batch_size=32, learning_rate=0.001
)
print("✓ Exercise router trained")

# Train per-exercise action classifiers
print("\nTraining per-exercise action classifiers...")
action_models = {}
for ex_idx, ex_id in enumerate(exercise_ids_train):
    ex_mask = y_exercise_train == ex_idx
    if ex_mask.sum() > 0:
        n_actions = len(np.unique(y_train[ex_mask]))
        action_models[ex_id] = fit_frequency_model(
            X_train[ex_mask], y_train[ex_mask], n_actions, device,
            epochs=150, batch_size=16, learning_rate=0.0005
        )
        print(f"  Exercise {ex_id}: {n_actions} actions")

print(f"\n✓ {len(action_models)} exercise-specific classifiers trained")

## 9. Evaluation: Cross-Subject Test Performance

In [ ]:
# Predict on test set using hierarchical routing
print("Predicting on held-out test subjects...")
y_pred = predict_hierarchical(
    exercise_model, action_models, exercise_ids_test, X_test, device
)

# Evaluate
metrics = classification_summary(y_test, y_pred)
print(f"\nCross-Subject Test Performance:")
print(f"  Accuracy: {metrics['accuracy']:.1%}")
print(f"  Balanced Accuracy: {metrics['balanced_accuracy']:.1%}")
print(f"  Macro F1: {metrics['macro_f1']:.1%}")

# Classification report
print(f"\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=classes, digits=3))

## 10. Confusion Matrix and Error Analysis

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot
fig, ax = plt.subplots(figsize=(12, 10))
plot_confusion_heatmap(
    cm, classes, ax=ax,
    title=f"Amputee Action Recognition: Cross-Subject Test\n(Accuracy: {metrics['accuracy']:.1%})"  
)
plt.tight_layout()
plt.savefig(results_dir / f"{EXPERIMENT_ID}_confusion.png", dpi=150, bbox_inches='tight')
plt.show()

print("✓ Confusion matrix saved")

## 11. Per-Action Performance

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred)

# Create per-action summary
action_performance = pd.DataFrame({
    'Action': classes,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Support': support
})

action_performance = action_performance.sort_values('F1-Score', ascending=False)
print("\nPer-Action Performance (sorted by F1):")
print(action_performance.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
plot_bar(
    action_performance.sort_values('F1-Score', ascending=True)['Action'],
    action_performance.sort_values('F1-Score', ascending=True)['F1-Score'],
    ax=ax,
    xlabel='Macro F1 Score', ylabel='Action',
    title='Per-Action Recognition Performance (Amputee Cohort)'
)
plt.tight_layout()
plt.savefig(results_dir / f"{EXPERIMENT_ID}_per_action_f1.png", dpi=150, bbox_inches='tight')
plt.show()

## 12. Translation Gap Analysis

Compare amputee performance to the able-bodied baseline to quantify the design challenge.

In [ ]:
# Load able-bodied results for comparison
import json

ablebody_result_path = results_dir / "EXP-008-ablebody-frequency-baseline_experiment.json"
if ablebody_result_path.exists():
    with open(ablebody_result_path) as f:
        ablebody_data = json.load(f)
    
    ablebody_f1 = ablebody_data['metrics']['macro_f1']
    ablebody_accuracy = ablebody_data['metrics']['accuracy']
    
    translation_gap_f1 = ablebody_f1 - metrics['macro_f1']
    translation_gap_acc = ablebody_accuracy - metrics['accuracy']
    
    print(f"\n### TRANSLATION GAP ###\n")
    print(f"Able-Bodied Baseline:")
    print(f"  Accuracy: {ablebody_accuracy:.1%}")
    print(f"  Macro F1: {ablebody_f1:.1%}")
    
    print(f"\nAmputee Reality:")
    print(f"  Accuracy: {metrics['accuracy']:.1%}")
    print(f"  Macro F1: {metrics['macro_f1']:.1%}")
    
    print(f"\nTranslation Loss:")
    print(f"  Accuracy drop: {translation_gap_acc:.1%}")
    print(f"  F1 drop: {translation_gap_f1:.1%}")
    
    print(f"\n### DESIGN IMPLICATIONS ###\n")
    if translation_gap_f1 > 0.3:
        print(f"⚠️  Large translation gap ({translation_gap_f1:.1%}):")
        print(f"   → EMG patterns differ substantially from intact kinematics.")
        print(f"   → May require prosthetic-specific training or simplified gesture set.")
    elif translation_gap_f1 > 0.1:
        print(f"⚠️  Moderate translation gap ({translation_gap_f1:.1%}):")
        print(f"   → Some actions transfer well, others need attention.")
        print(f"   → Use Notebook 3 to identify which patterns are stable.")
    else:
        print(f"✓ Small translation gap ({translation_gap_f1:.1%}):")
        print(f"   → EMG patterns largely capture motor intent from intact baseline.")
else:
    print("⚠️  Notebook 1 results not found. Run it first to enable comparison.")

## 13. Save Results

In [ ]:
# Save experiment summary
experiment_data = {
    'experiment_id': EXPERIMENT_ID,
    'cohort': COHORT,
    'timestamp': datetime.now().isoformat(),
    'seed': SEED,
    'test_subjects': sorted(test_subjects),
    'metrics': {
        'accuracy': float(metrics['accuracy']),
        'balanced_accuracy': float(metrics['balanced_accuracy']),
        'macro_f1': float(metrics['macro_f1']),
    },
    'per_action': action_performance.to_dict(orient='records'),
    'n_windows_train': int(X_train.shape[0]),
    'n_windows_test': int(X_test.shape[0]),
    'n_unique_actions': len(classes),
    'feature_dimension': int(X.shape[1]),
}

save_experiment_json(results_dir, EXPERIMENT_ID, experiment_data)
print(f"✓ Experiment results saved: {EXPERIMENT_ID}")
print(f"\n### NEXT STEP ###")
print(f"Run Notebook 3 to analyze which patterns transfer using Responsible AI.")